<a href="https://colab.research.google.com/github/AMIRTHA192/ITA0601-ML/blob/main/EX_3_AND_EX_4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd
import math
from collections import Counter

# Training dataset: Play Tennis
data = pd.DataFrame({
    'Outlook': ['Sunny', 'Sunny', 'Overcast', 'Rain', 'Rain',
                'Rain', 'Overcast', 'Sunny', 'Sunny', 'Rain',
                'Sunny', 'Overcast', 'Overcast', 'Rain'],
    'Temperature': ['Hot', 'Hot', 'Hot', 'Mild', 'Cool',
                    'Cool', 'Cool', 'Mild', 'Cool', 'Mild',
                    'Mild', 'Mild', 'Hot', 'Mild'],
    'Humidity': ['High', 'High', 'High', 'High', 'Normal',
                 'Normal', 'Normal', 'High', 'Normal', 'Normal',
                 'Normal', 'High', 'Normal', 'High'],
    'Wind': ['Weak', 'Strong', 'Weak', 'Weak', 'Weak',
             'Strong', 'Strong', 'Weak', 'Weak', 'Weak',
             'Strong', 'Strong', 'Weak', 'Strong'],
    'Play': ['No', 'No', 'Yes', 'Yes', 'Yes',
             'No', 'Yes', 'No', 'Yes', 'Yes',
             'Yes', 'Yes', 'Yes', 'No']
})

# Calculate entropy
def entropy(labels):
    counts = Counter(labels)
    total = len(labels)
    return -sum((count / total) * math.log2(count / total)
                for count in counts.values())

# ID3 algorithm
def id3(df, target, features):
    if len(df[target].unique()) == 1:
        return df[target].iloc[0]

    if not features:
        return df[target].mode()[0]

    gains = {}
    parent_entropy = entropy(df[target])

    for feature in features:
        weighted_entropy = 0
        for value in df[feature].unique():
            subset = df[df[feature] == value]
            weighted_entropy += (len(subset) / len(df)) * entropy(subset[target])
        gains[feature] = parent_entropy - weighted_entropy

    best = max(gains, key=gains.get)
    tree = {best: {}}

    for value in df[best].unique():
        subset = df[df[best] == value]
        if subset.empty:
            tree[best][value] = df[target].mode()[0]
        else:
            tree[best][value] = id3(
                subset, target, [f for f in features if f != best]
            )

    return tree

# Predict a new sample
def predict(tree, sample, default='Yes'):
    if not isinstance(tree, dict):
        return tree

    feature = next(iter(tree))
    value = sample.get(feature)

    if value not in tree[feature]:
        return default

    return predict(tree[feature][value], sample, default)

features = ['Outlook', 'Temperature', 'Humidity', 'Wind']
tree = id3(data, 'Play', features)

print("Decision Tree:")
print(tree)

sample = {
    'Outlook': 'Overcast',
    'Temperature': 'Cool',
    'Humidity': 'High',
    'Wind': 'Strong'
}

print("\nNew Sample:", sample)
print("Predicted Class:", predict(tree, sample))

Decision Tree:
{'Outlook': {'Sunny': {'Humidity': {'High': 'No', 'Normal': 'Yes'}}, 'Overcast': 'Yes', 'Rain': {'Wind': {'Weak': 'Yes', 'Strong': 'No'}}}}

New Sample: {'Outlook': 'Overcast', 'Temperature': 'Cool', 'Humidity': 'High', 'Wind': 'Strong'}
Predicted Class: Yes


In [2]:
import numpy as np
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import OneHotEncoder

# Load dataset
iris = load_iris()
X = iris.data
y = iris.target

# Normalize input
scaler = StandardScaler()
X = scaler.fit_transform(X)

# Convert target into one-hot encoding
encoder = OneHotEncoder(sparse_output=False)
Y = encoder.fit_transform(y.reshape(-1, 1))

# Split dataset
X_train, X_test, Y_train, Y_test = train_test_split(
    X, Y, test_size=0.2, random_state=42, stratify=y
)

# Initialize network
np.random.seed(42)
input_size = 4
hidden_size = 8
output_size = 3
learning_rate = 0.1

W1 = np.random.randn(input_size, hidden_size) * 0.1
b1 = np.zeros((1, hidden_size))
W2 = np.random.randn(hidden_size, output_size) * 0.1
b2 = np.zeros((1, output_size))

def sigmoid(x):
    return 1 / (1 + np.exp(-np.clip(x, -500, 500)))

def softmax(x):
    exp_x = np.exp(x - np.max(x, axis=1, keepdims=True))
    return exp_x / np.sum(exp_x, axis=1, keepdims=True)

# Train using backpropagation
for epoch in range(2000):
    # Forward propagation
    hidden = sigmoid(np.dot(X_train, W1) + b1)
    output = softmax(np.dot(hidden, W2) + b2)

    # Output error
    error = output - Y_train

    # Backpropagation
    dW2 = np.dot(hidden.T, error) / len(X_train)
    db2 = np.mean(error, axis=0, keepdims=True)

    hidden_error = np.dot(error, W2.T) * hidden * (1 - hidden)
    dW1 = np.dot(X_train.T, hidden_error) / len(X_train)
    db1 = np.mean(hidden_error, axis=0, keepdims=True)

    # Update weights and biases
    W2 -= learning_rate * dW2
    b2 -= learning_rate * db2
    W1 -= learning_rate * dW1
    b1 -= learning_rate * db1

# Test network
hidden = sigmoid(np.dot(X_test, W1) + b1)
output = softmax(np.dot(hidden, W2) + b2)

predictions = np.argmax(output, axis=1)
actual = np.argmax(Y_test, axis=1)
accuracy = np.mean(predictions == actual) * 100

print("Actual Classes:   ", actual)
print("Predicted Classes:", predictions)
print("Test Accuracy: {:.2f}%".format(accuracy))

# Classify one new sample
new_sample = np.array([[5.1, 3.5, 1.4, 0.2]])
new_sample = scaler.transform(new_sample)

hidden = sigmoid(np.dot(new_sample, W1) + b1)
output = softmax(np.dot(hidden, W2) + b2)
predicted_class = np.argmax(output, axis=1)[0]

print("\nNew Sample Prediction:", iris.target_names[predicted_class])

Actual Classes:    [0 2 1 1 0 1 0 0 2 1 2 2 2 1 0 0 0 1 1 2 0 2 1 2 2 1 1 0 2 0]
Predicted Classes: [0 2 1 1 0 1 0 0 2 1 2 2 2 1 0 0 0 1 1 2 0 2 1 2 2 2 1 0 2 0]
Test Accuracy: 96.67%

New Sample Prediction: setosa
